# 01. Depuración y transformación de datos

En este notebook preparo los datos que después voy a usar en `02_analisis_narrativa_presentacion.ipynb`. 

____________

## 1. Librerías, carpetas y funciones simples

Primero importo las librerías y dejo preparadas las carpetas.

In [1]:
import os, json, re, unicodedata
import numpy as np
import pandas as pd
from pyproj import Transformer


# Si ejecuto el notebook desde la carpeta notebooks, subo a la raíz del proyecto.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

RAW='datos/raw'; INTERIM='datos/interim'; PROCESSED='datos/processed'; ANALISIS='datos/processed/analisis'; INFORMES='informes'
os.makedirs(INTERIM, exist_ok=True); os.makedirs(PROCESSED, exist_ok=True); os.makedirs(ANALISIS, exist_ok=True)

ESPECIALES = {'', '..', '-', 'NC', 'N/C', 'No consta', 'no consta', 'No Consta'}
def limpiar_texto(s):
    if pd.isna(s): return np.nan
    t = ' '.join(str(s).strip().split())
    while t.startswith('"'): t=t[1:]
    while t.endswith('"'): t=t[:-1]
    t=t.replace('""','"').strip()
    if t in ESPECIALES: return np.nan
    return t

def normalizar_codigo(s):
    t=limpiar_texto(s)
    if pd.isna(t): return np.nan
    if str(t).isdigit(): return str(int(t))
    try:
        f=float(str(t).replace(',','.'))
        if f.is_integer(): return str(int(f))
    except Exception: pass
    return str(t)

def normalizar_nombre(texto):
    texto=str(texto).lower().strip()
    texto=unicodedata.normalize('NFKD', texto)
    texto=''.join(x for x in texto if not unicodedata.combining(x))
    texto=re.sub(r'[^a-z0-9]+',' ',texto)
    return re.sub(r'\s+',' ',texto).strip()

def read_csv_str(path):
    df=pd.read_csv(path, dtype=str, encoding='utf-8-sig', low_memory=False)
    df.columns=[limpiar_texto(c) for c in df.columns]
    for c in df.columns:
        if df[c].dtype=='object':
            df[c]=df[c].map(limpiar_texto)
    return df

## 2. Tabla territorial

Creo una tabla sencilla de barrios y distritos a partir del GeoJSON oficial. Esta tabla me sirve para tener una referencia común de `codi_barri`.

In [2]:
# tabla territorial
geo=json.load(open(os.path.join(RAW,'barcelona_barrios_unidades_administrativas_poligonos.json'),encoding='utf-8'))
barrios=[]; distritos=[]
for feat in geo['features']:
    p=feat.get('properties',{})
    if p.get('TIPUS_UA')=='BARRI':
        barrios.append({'codi_barri':normalizar_codigo(p.get('BARRI')), 'nom_barri':limpiar_texto(p.get('NOM')), 'codi_districte':normalizar_codigo(p.get('DISTRICTE'))})
    if p.get('TIPUS_UA')=='DISTRICTE':
        distritos.append({'codi_districte':normalizar_codigo(p.get('DISTRICTE')), 'nom_districte':limpiar_texto(p.get('NOM'))})
tabla_maestra=pd.DataFrame(barrios).merge(pd.DataFrame(distritos), on='codi_districte', how='left')
tabla_maestra=tabla_maestra.sort_values(['codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce')).reset_index(drop=True)
tabla_maestra.to_csv(os.path.join(INTERIM,'tabla_maestra_barrios.csv'), index=False)

## 3. Limpieza de IRIS

Leo todos los CSV de IRIS, limpio comillas residuales y dejo solo las columnas que necesito para el análisis. También elimino duplicados por año de fichero y `fitxa_id`, que era la regla usada hasta ahora.

In [3]:
# IRIS limpio
iris_files=sorted([f for f in os.listdir(RAW) if f.startswith('iris_incidencias_ciudadanas_20') and f.endswith('.csv')])
frames=[]                                           
for f in iris_files:
    df=read_csv_str(os.path.join(RAW,f))
    df['ANY_FITXER']=f.split('_')[-1].replace('.csv','')
    frames.append(df)
iris=pd.concat(frames, ignore_index=True)
iris_limpio=pd.DataFrame({
    'fitxa_id': iris['FITXA_ID'].map(limpiar_texto),
    'tipus': iris['TIPUS'].map(limpiar_texto),
    'area': iris['AREA'].map(limpiar_texto),
    'element': iris['ELEMENT'].map(limpiar_texto),
    'detall': iris['DETALL'].map(limpiar_texto),
    'any': iris['ANY_DATA_ALTA'].map(normalizar_codigo),
    'mes_data_alta': iris['MES_DATA_ALTA'].map(normalizar_codigo),
    'codi_districte': iris['CODI_DISTRICTE'].map(normalizar_codigo),
    'districte': iris['DISTRICTE'].map(limpiar_texto),
    'codi_barri': iris['CODI_BARRI'].map(normalizar_codigo),
    'barri': iris['BARRI'].map(limpiar_texto),
    'seccio_censal': iris['SECCIO_CENSAL'].map(normalizar_codigo),
    'suport': iris['SUPORT'].map(limpiar_texto),
    'canals_resposta': iris['CANALS_RESPOSTA'].map(limpiar_texto),
    'any_fitxer': iris['ANY_FITXER'].map(normalizar_codigo),
})
iris_limpio=iris_limpio.drop_duplicates(subset=['any_fitxer', 'fitxa_id'], keep='first')
iris_limpio.to_csv(os.path.join(INTERIM,'iris_2015_2025_limpio_parcial.csv'), index=False)

## 4. Limpieza del bloque inmobiliario

Leo las series inmobiliarias, normalizo nombres de columnas, códigos y valores numéricos. 

In [4]:
# inmobiliario limpio
def cargar_serie(prefijo, excluir_2026=True):
    files=sorted([f for f in os.listdir(RAW) if f.startswith(prefijo) and f.endswith('.csv') and not (excluir_2026 and f.endswith('_2026.csv'))])
    out=[]
    for f in files:
        df=read_csv_str(os.path.join(RAW,f)); df['archivo_origen']=f; out.append(df)
    return pd.concat(out, ignore_index=True)

def to_num(s): return pd.to_numeric(s.astype(str).str.replace(',','.', regex=False), errors='coerce')

valor=cargar_serie('vivienda_valor_catastral_')
valor_l=pd.DataFrame({
'any':valor['Any'].map(normalizar_codigo),'codi_districte':valor['Codi_districte'].map(normalizar_codigo),'nom_districte':valor['Nom_districte'].map(limpiar_texto),'codi_barri':valor['Codi_barri'].map(normalizar_codigo),'nom_barri':valor['Nom_barri'].map(limpiar_texto),'seccio_censal':valor['Seccio_censal'].map(normalizar_codigo),'desc_valor':valor['Desc_valors'].map(limpiar_texto),'valor':to_num(valor['Valors']),'archivo_origen':valor['archivo_origen']})
valor_l.to_csv(os.path.join(INTERIM,'vivienda_valor_catastral_2018_2025_limpio_parcial.csv'), index=False)

num=cargar_serie('compraventas_notariales_numero_por_uso_')
num_l=pd.DataFrame({'any':num['Any'].map(normalizar_codigo),'mes':num['Mes'].map(normalizar_codigo),'codi_districte':num['Codi_Districte'].map(normalizar_codigo),'nom_districte':num['Nom_Districte'].map(limpiar_texto),'codi_barri':num['Codi_Barri'].map(normalizar_codigo),'nom_barri':num['Nom_Barri'].map(limpiar_texto),'tipologia_uso_codi':num['Tipologia_Us_Codi'].map(normalizar_codigo),'tipologia_uso_desc':num['Tipologia_Us_Desc'].map(limpiar_texto),'numero_compraventas':pd.to_numeric(num['Nombre'], errors='coerce'),'archivo_origen':num['archivo_origen']})
num_l.to_csv(os.path.join(INTERIM,'compraventas_notariales_numero_por_uso_2012_2025_limpio_parcial.csv'), index=False)

sup=cargar_serie('compraventas_notariales_superficie_por_uso_')
sup_l=pd.DataFrame({'any':sup['Any'].map(normalizar_codigo),'mes':sup['Mes'].map(normalizar_codigo),'codi_districte':sup['Codi_Districte'].map(normalizar_codigo),'nom_districte':sup['Nom_Districte'].map(limpiar_texto),'codi_barri':sup['Codi_Barri'].map(normalizar_codigo),'nom_barri':sup['Nom_Barri'].map(limpiar_texto),'tipologia_uso_codi':sup['Tipologia_Us_Codi'].map(normalizar_codigo),'tipologia_uso_desc':sup['Tipologia_Us_Desc'].map(limpiar_texto),'superficie_m2':to_num(sup['Superficie_m2']),'archivo_origen':sup['archivo_origen']})
sup_l.to_csv(os.path.join(INTERIM,'compraventas_notariales_superficie_por_uso_2012_2025_limpio_parcial.csv'), index=False)

cuota=cargar_serie('cuota_catastral_por_uso_')
cuota_l=pd.DataFrame({'any':cuota['Any'].map(normalizar_codigo),'codi_districte':cuota['Codi_districte'].map(normalizar_codigo),'nom_districte':cuota['Nom_districte'].map(limpiar_texto),'codi_barri':cuota['Codi_barri'].map(normalizar_codigo),'nom_barri':cuota['Nom_barri'].map(limpiar_texto),'seccio_censal':cuota['Seccio_censal'].map(normalizar_codigo),'desc_uso':cuota['Desc_us'].map(limpiar_texto),'desc_quota':cuota['Desc_quota'].map(limpiar_texto),'valor_eur':to_num(cuota['Valor_€']),'archivo_origen':cuota['archivo_origen']})
cuota_l.to_csv(os.path.join(INTERIM,'cuota_catastral_por_uso_2018_2025_limpio_parcial.csv'), index=False)

## 5. Agregación por barrio y año

Ahora agrego las tablas limpias a una fila por barrio y año. Uso `groupby`, `pivot_table` y `merge`, que son las operaciones principales de pandas que necesito para este proyecto.

In [5]:
# agregaciones
# Objetivo de esta celda: pasar de registros detallados a tablas resumen con una fila por año y barrio. Estas tablas son más cómodas para cruzar
# después con la tabla maestra territorial y con el resto de indicadores.
MAPA_TIPUS={'INCIDENCIA':'total_incidencias_iris','CONSULTA':'total_consultas_iris','QUEIXA':'total_quejas_iris',
            'SUGGERIMENT':'total_sugerencias_iris','PETICIO DE SERVEI':'total_peticiones_servicio_iris','AGRAIMENT':'total_agradecimientos_iris'}

# IRIS: me quedo solo con registros que tienen año y barrio, y limito el periodo al rango que voy a analizar.
iris_a=iris_limpio.dropna(subset=['any','codi_barri']).copy(); iris_a=iris_a[iris_a['any'].astype(int).between(2015,2025)]

# Agrupo por año y barrio: 
base_ids=iris_a.groupby(['any','codi_barri'], as_index=False).agg(nom_barri=('barri','first'), codi_districte=('codi_districte','first'), nom_districte=('districte','first'), total_registros_iris=('fitxa_id','size'))

# Con pivot_table convierto los tipos de solicitud en columnas separadas:incidencias, consultas, quejas, sugerencias, etc.

piv=iris_a.pivot_table(index=['any','codi_barri'], columns='tipus', values='fitxa_id', aggfunc='size', fill_value=0).reset_index()
piv=piv.rename(columns=MAPA_TIPUS)
for c in MAPA_TIPUS.values():
    if c not in piv.columns:
        piv[c] = 0
# Uno el total general con las columnas por tipo de solicitud.
iris_ag=base_ids.merge(piv[['any','codi_barri']+list(MAPA_TIPUS.values())], on=['any','codi_barri'], how='left')
for c in MAPA_TIPUS.values():
    if c not in iris_ag: iris_ag[c]=0
    iris_ag[c]=iris_ag[c].fillna(0).astype(int)
iris_ag=iris_ag.sort_values(['any','codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce'))
iris_ag.to_csv(os.path.join(INTERIM,'iris_2015_2025_agregado_barrio_anio.csv'), index=False)

# Valor catastral

map_val={"Valor_cadastral_total_€":"valor_catastral_total_eur_sum","Valor_sòl_total_€":"valor_suelo_total_eur_sum","Valor_construcció_més_serveis_total_€":"valor_construccion_mas_servicios_total_eur_sum"}
map_mean={"Valor_cadastral_unitari_€/m2":"valor_catastral_unitario_eur_m2_mean","Valor_sòl_unitari_€/m2":"valor_suelo_unitario_eur_m2_mean","Valor_construcció_més_serveis_unitari_€/m2":"valor_construccion_mas_servicios_unitario_eur_m2_mean"}
v=valor_l.dropna(subset=['any','codi_barri']).copy()
# Mantengo una base identificativa y cuento cuántas secciones censales aportan información en cada barrio-año.

ident=v.groupby(['any','codi_barri'], as_index=False).agg(nom_barri=('nom_barri','first'), codi_districte=('codi_districte','first'), nom_districte=('nom_districte','first'), cantidad_secciones_censales_informadas=('seccio_censal', pd.Series.nunique))
sums=v[v['desc_valor'].isin(map_val)].pivot_table(index=['any','codi_barri'], columns='desc_valor', values='valor', aggfunc='sum', fill_value=0).reset_index().rename(columns=map_val)
means=v[v['desc_valor'].isin(map_mean)].pivot_table(index=['any','codi_barri'], columns='desc_valor', values='valor', aggfunc='mean').reset_index().rename(columns=map_mean)

# Vuelvo a unir identificadores, sumas y medias en una sola tabla agregada.
valor_ag=ident.merge(sums,on=['any','codi_barri'],how='left').merge(means,on=['any','codi_barri'],how='left')

for c in list(map_val.values())+list(map_mean.values()):
    if c not in valor_ag: valor_ag[c]=np.nan
valor_ag=valor_ag.sort_values(['any','codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce'))
valor_ag.to_csv(os.path.join(INTERIM,'vivienda_valor_catastral_2018_2025_agregado_barrio_anio.csv'), index=False)

# Compraventas: traduzco los códigos de uso a nombres legibles y reutilizo la misma función para número de operaciones y superficie vendida.
uso={'1':'residencial','2':'aparcamiento','3':'comercial','4':'oficina','5':'turistico','6':'solares','7':'industrial','8':'equipamientos','9':'sin_tipologia'}

def agregar_uso(df, valor_col, total_col, prefix, suffix, out):
    # La función recibe una tabla detallada y devuelve una tabla barrio-año. Además del total anual, crea columnas por uso: residencial, comercial, etc.
    d=df.dropna(subset=['any','codi_barri']).copy(); d['uso']=d['tipologia_uso_codi'].map(uso).fillna('sin_tipologia')
    ident=d.groupby(['any','codi_barri'],as_index=False).agg(nom_barri=('nom_barri','first'), codi_districte=('codi_districte','first'), nom_districte=('nom_districte','first'))
    total=d.groupby(['any','codi_barri'],as_index=False)[valor_col].sum().rename(columns={valor_col:total_col})
    
    # Pivot: cada uso pasa de ser una categoría en filas a una columna numérica.
    piv=d.pivot_table(index=['any','codi_barri'], columns='uso', values=valor_col, aggfunc='sum', fill_value=0).reset_index()
    piv=piv.rename(columns={u:f'{prefix}_{u}_{suffix}' for u in uso.values()})
    res=ident.merge(total,on=['any','codi_barri'],how='left').merge(piv,on=['any','codi_barri'],how='left')
    for u in uso.values():
        col=f'{prefix}_{u}_{suffix}'
        if col not in res: res[col]=0
    res=res.sort_values(['any','codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce'))
    res.to_csv(os.path.join(INTERIM,out), index=False)
    return res
num_ag=agregar_uso(num_l,'numero_compraventas','total_compraventas_anual','compraventas','anual','compraventas_notariales_numero_por_uso_2012_2025_agregado_barrio_anio.csv')
sup_ag=agregar_uso(sup_l,'superficie_m2','total_superficie_compraventas_m2_anual','superficie_compraventas','m2_anual','compraventas_notariales_superficie_por_uso_2012_2025_agregado_barrio_anio.csv')

# Cuota catastral: normalizo nombres de usos y tipos de cuota para construir columnas consistentes como cuota_liquida_residencial_eur_sum.
uso_cuota={"Residencial":"residencial","Comerç":"comercio","Aparcament":"aparcamiento",
           "Oficines":"oficinas","Solars":"solares","Ensenyament i Cultura":"ensenyanza_cultura",
           "Turisme i Hostaleria":"turismo_hosteleria","Sanitat":"sanidad","Indústria":"industria",
           "Esportiu":"deportivo","Religiós":"religioso","Administracions Publiques":"administraciones_publicas",
           "Espectacles":"espectaculos"}

tipo_cuota={"Quota_líquida":"liquida","Quota_íntegra":"integra","Quota_total":"total"}
c=cuota_l.dropna(subset=['any','codi_barri']).copy(); c['tipo']=c['desc_quota'].map(tipo_cuota); c['uso']=c['desc_uso'].map(uso_cuota).fillna('sin_uso')

# Primero calculo totales por tipo de cuota y después el detalle por tipo+uso.
ident=c.groupby(['any','codi_barri'],as_index=False).agg(nom_barri=('nom_barri','first'), codi_districte=('codi_districte','first'), nom_districte=('nom_districte','first'))
ctotal=c.pivot_table(index=['any','codi_barri'], columns='tipo', values='valor_eur', aggfunc='sum', fill_value=0).reset_index().rename(columns={t:f'cuota_{t}_total_eur_sum' for t in tipo_cuota.values()})

# Creo el nombre final de cada columna antes de pivotar.
c['col']='cuota_'+c['tipo']+'_'+c['uso']+'_eur_sum'
cpiv=c.pivot_table(index=['any','codi_barri'], columns='col', values='valor_eur', aggfunc='sum', fill_value=0).reset_index()

# Integro totales y detalle por uso en la tabla final de cuota catastral.
cuota_ag=ident.merge(ctotal,on=['any','codi_barri'],how='left').merge(cpiv,on=['any','codi_barri'],how='left')
for t in tipo_cuota.values():
    if f'cuota_{t}_total_eur_sum' not in cuota_ag: cuota_ag[f'cuota_{t}_total_eur_sum']=0
    for u in list(uso_cuota.values())+['sin_uso']:
        col=f'cuota_{t}_{u}_eur_sum'
        if col not in cuota_ag: cuota_ag[col]=0
cuota_ag=cuota_ag.sort_values(['any','codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce'))
cuota_ag.to_csv(os.path.join(INTERIM,'cuota_catastral_por_uso_2018_2025_agregado_barrio_anio.csv'), index=False)

In [6]:
cpiv

col,any,codi_barri,cuota_integra_administraciones_publicas_eur_sum,cuota_integra_aparcamiento_eur_sum,cuota_integra_comercio_eur_sum,cuota_integra_deportivo_eur_sum,cuota_integra_ensenyanza_cultura_eur_sum,cuota_integra_espectaculos_eur_sum,cuota_integra_industria_eur_sum,cuota_integra_oficinas_eur_sum,...,cuota_total_ensenyanza_cultura_eur_sum,cuota_total_espectaculos_eur_sum,cuota_total_industria_eur_sum,cuota_total_oficinas_eur_sum,cuota_total_religioso_eur_sum,cuota_total_residencial_eur_sum,cuota_total_sanidad_eur_sum,cuota_total_sin_uso_eur_sum,cuota_total_solares_eur_sum,cuota_total_turismo_hosteleria_eur_sum
0,2018,1,335369.64,494104.15,1663505.90,48382.89,1128153.94,414801.34,4273.27,680848.00,...,532859.10,81030.76,4605.07,736600.56,15317.41,5426519.10,23835.73,0.0,44612.75,2673696.65
1,2018,10,30796.10,928164.32,1730633.87,92348.01,193720.15,59889.18,34394.96,848004.32,...,54603.63,51133.57,36063.21,834020.44,5202.14,10592255.57,27185.34,0.0,11314.14,960517.22
2,2018,11,331177.13,498287.18,937492.66,1658505.38,773530.11,160535.62,135567.54,686801.84,...,240227.33,61751.45,83775.68,739404.81,1128.54,5783701.70,154791.57,0.0,31642.45,1373329.71
3,2018,12,126815.07,508176.36,756735.91,27436.20,141009.21,0.00,2805716.16,2645091.90,...,118514.47,0.00,3245358.64,3033549.16,0.00,222746.57,4886.66,0.0,5025490.36,29433.84
4,2018,13,7535.48,672482.67,758712.31,78841.11,152610.95,454.38,9183.22,684122.14,...,26657.23,0.00,10578.10,679171.62,1220.99,4643412.52,181288.03,0.0,159136.97,1433866.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,2025,71,34622.13,324021.23,441751.98,11962.97,628173.53,0.00,551950.95,2210613.03,...,616897.21,0.00,569618.52,2331453.38,0.00,4664155.62,12113.01,0.0,447718.25,214078.18
580,2025,72,50503.07,219225.53,415057.04,109917.46,247138.05,0.00,0.00,74766.15,...,28779.66,0.00,0.00,70667.92,29043.52,4972636.23,16446.93,0.0,544.60,17439.96
581,2025,73,0.00,217930.61,393160.53,31807.39,98433.70,0.00,488482.91,56041.82,...,9943.13,0.00,560266.41,57196.80,164.10,4937423.41,0.00,0.0,98951.57,0.00
582,2025,8,62948.02,1425220.34,4043040.96,5203.57,865976.16,9810.22,57928.08,3882878.05,...,238733.81,11387.75,32408.35,3791970.55,30682.76,22843844.30,174467.36,0.0,18562.30,2754076.14


## 6. Tabla base final

Construyo la tabla base con todos los barrios y años disponibles. Después voy añadiendo cada bloque con `merge` y guardo el CSV que consume el notebook 02.

In [7]:
# tabla base
all_years=sorted(set(pd.concat([iris_ag['any'], valor_ag['any'], num_ag['any'], sup_ag['any'], cuota_ag['any']]).dropna().astype(int)))
panel=tabla_maestra.assign(key=1).merge(pd.DataFrame({'any':all_years,'key':1}),on='key').drop(columns='key')
panel=panel[['any','codi_barri','nom_barri','codi_districte','nom_districte']]
panel=panel.sort_values(['any','codi_districte','codi_barri'], key=lambda s: pd.to_numeric(s, errors='coerce')).reset_index(drop=True)
blocks=[('iris',iris_ag,2015,2025),('valor_catastral',valor_ag,2018,2025),('compraventas_numero',num_ag,2012,2025),('compraventas_superficie',sup_ag,2012,2025),('cuota_catastral',cuota_ag,2018,2025)]
tabla=panel.copy()

for name,df,amin,amax in blocks:
    tabla[f'tiene_cobertura_{name}']=tabla['any'].between(amin,amax).astype(int)
    add=df.drop(columns=[x for x in ['nom_barri','codi_districte','nom_districte'] if x in df.columns]).copy()
    add['any']=pd.to_numeric(add['any'], errors='coerce').astype(int)
    add['codi_barri']=add['codi_barri'].astype(str)
    tabla['codi_barri']=tabla['codi_barri'].astype(str)
    tabla=tabla.merge(add,on=['any','codi_barri'],how='left')

# extracc de nomb col old
old_cols=pd.read_csv(os.path.join(PROCESSED,'tabla_base_barrio_anio.csv'), nrows=0).columns.tolist()
for col in old_cols:
    if col not in tabla.columns: tabla[col]=np.nan
tabla=tabla[old_cols]
tabla.to_csv(os.path.join(PROCESSED,'tabla_base_barrio_anio.csv'), index=False)

dic=pd.DataFrame({'variable':tabla.columns,'bloque_origen':'','descripcion':'','clave_union':['si' if c in ['any','codi_barri'] 
                                                                                              else 'no' for c in tabla.columns],'rango_temporal_bloque':'','cantidad_nulos_tabla_base':[tabla[c].isna().sum() for c in tabla.columns],'nulos_esperables':''})
dic.to_csv(os.path.join(PROCESSED,'diccionario_variables_tabla_base.csv'), index=False) 

## 7. Tablas analíticas para el notebook 02

Genero las tablas específicas de población, densidad, mapas, tipologías y equidad. Así el notebook 02 puede centrarse en gráficos y narrativa.

In [8]:
# Tablas analíticas para el notebook 02
# geojson wgs84
transformer=Transformer.from_crs('EPSG:25831','EPSG:4326',always_xy=True)
def trans(coords):
    if isinstance(coords[0], (int,float)):
        lon,lat=transformer.transform(coords[0],coords[1]); return [lon,lat]
    return [trans(x) for x in coords]
features=[]
for feat in geo['features']:
    p=feat.get('properties',{})
    if p.get('TIPUS_UA')=='BARRI':
        nom=p.get('NOM','')
        features.append({'type':'Feature','properties':{'codi_barri':str(int(p.get('BARRI'))),
                                                        'codi_districte':str(int(p.get('DISTRICTE'))),
                                                        'nom_barri':nom,'nom_barri_norm':normalizar_nombre(nom)},
                                                        'geometry':{'type':feat['geometry']['type'],
                                                        'coordinates':trans(feat['geometry']['coordinates'])}})

open(os.path.join(ANALISIS,'barrios_wgs84.geojson'),'w',encoding='utf-8').write(json.dumps({'type':'FeatureCollection','features':features},ensure_ascii=False))

1214911

## 8. Población y densidad

Uno la tabla base con población y densidad oficial para poder calcular incidencias por 1.000 habitantes.

In [9]:
# población densidad
pframes=[]
for f in sorted(os.listdir(os.path.join(RAW,'poblacion_sexo_barrios_2015_2024'))):
    if f.endswith('.csv'):
        df=pd.read_csv(os.path.join(RAW,'poblacion_sexo_barrios_2015_2024',f))
        df['any']=pd.to_datetime(df['Data_Referencia']).dt.year.astype(int); df['codi_barri']=df['Codi_Barri'].astype(str); df['codi_districte']=df['Codi_Districte'].astype(str)
        pframes.append(df.groupby(['any','codi_districte','Nom_Districte','codi_barri','Nom_Barri'],as_index=False)['Valor'].sum().rename(columns={'Nom_Districte':'nom_districte','Nom_Barri':'nom_barri','Valor':'poblacion_total'}))
poblacion_barrios=pd.concat(pframes,ignore_index=True).sort_values(['any','codi_barri'])
poblacion_ciudad=poblacion_barrios.groupby('any',as_index=False)['poblacion_total'].sum().rename(columns={'poblacion_total':'poblacion_barcelona'})
dframes=[]
for f in sorted(os.listdir(os.path.join(RAW,'densidad_barrios_2015_2021'))):
    if f.endswith('.csv'):
        df=pd.read_csv(os.path.join(RAW,'densidad_barrios_2015_2021',f)).rename(columns={'Any':'any','Codi_Districte':'codi_districte','Nom_Districte':'nom_districte',
                                                                                         'Codi_Barri':'codi_barri','Nom_Barri':'nom_barri','Població':'poblacion_oficial_densidad','Superfície (ha)':'superficie_ha',
                                                                                         'Superfície Residencial (ha)':'superficie_residencial_ha','Densitat (hab/ha)':'densidad_hab_ha',
                                                                                         'Densitat neta (hab/ha)':'densidad_neta_hab_ha'})
        df['codi_barri']=df['codi_barri'].astype(str); df['codi_districte']=df['codi_districte'].astype(str); dframes.append(df)
densidad=pd.concat(dframes,ignore_index=True).sort_values(['any','codi_barri'])
sup2021=densidad[densidad['any']==2021][['codi_barri','superficie_ha']].drop_duplicates('codi_barri').rename(columns={'superficie_ha':'superficie_ha_base_2021'})
tabla2=tabla.copy(); tabla2['codi_barri']=tabla2['codi_barri'].astype(str); tabla2['codi_districte']=tabla2['codi_districte'].astype(str); tabla2['any']=tabla2['any'].astype(int)
tabla_2015_2024=tabla2[tabla2['any'].between(2015,2024)].copy()
limpieza=iris_limpio[(iris_limpio['tipus']=='INCIDENCIA')&(iris_limpio['area']=="Recollida i neteja de l'espai urbà")&(iris_limpio['element']=='Neteja carrers i/o places')].copy()
limpieza['any']=pd.to_numeric(limpieza['any'],errors='coerce'); limpieza=limpieza[limpieza['any'].between(2015,2024)]
limpieza_barrios=limpieza.groupby(['any','codi_barri'],as_index=False).size().rename(columns={'size':'incidencias_limpieza'}); limpieza_barrios['any']=limpieza_barrios['any'].astype(int)

df_analisis=tabla_2015_2024.merge(poblacion_barrios[['any','codi_barri','poblacion_total']],on=['any','codi_barri'],
                                  how='left').merge(densidad[['any','codi_barri','densidad_hab_ha','superficie_ha']],on=['any','codi_barri'],
                                  how='left').merge(sup2021,on='codi_barri',how='left').merge(limpieza_barrios,on=['any','codi_barri'],how='left')

for col in ['total_incidencias_iris','poblacion_total','densidad_hab_ha','superficie_ha','superficie_ha_base_2021','incidencias_limpieza']:
    df_analisis[col]=pd.to_numeric(df_analisis[col],errors='coerce')
df_analisis['incidencias_limpieza']=df_analisis['incidencias_limpieza'].fillna(0)
df_analisis['densidad_estimada_hab_ha']=df_analisis['poblacion_total']/df_analisis['superficie_ha_base_2021']
df_analisis['densidad_analitica_hab_ha']=df_analisis['densidad_hab_ha']
mas=df_analisis['any'].between(2022,2024); df_analisis.loc[mas,'densidad_analitica_hab_ha']=df_analisis.loc[mas,'densidad_estimada_hab_ha']
df_analisis['fuente_densidad']=np.where(mas,'Estimación con superficie oficial 2021','Dato oficial Open Data')
df_analisis['incidencias_por_1000_hab']=df_analisis['total_incidencias_iris']/df_analisis['poblacion_total']*1000
df_analisis['incidencias_limpieza_por_1000_hab']=df_analisis['incidencias_limpieza']/df_analisis['poblacion_total']*1000
df_analisis['perfil_urbano']=np.where(df_analisis['codi_districte'].isin(['1','2','6']),'Centro','Periferia')
df_2024=df_analisis[df_analisis['any']==2024].copy()

base15=df_analisis[df_analisis['any']==2015][['codi_barri','nom_barri','nom_districte','perfil_urbano','poblacion_total','total_incidencias_iris','incidencias_por_1000_hab']].rename(
    columns={'poblacion_total':'poblacion_2015','total_incidencias_iris':'incidencias_2015','incidencias_por_1000_hab':'incidencias_1000_2015'})
base24=df_2024[['codi_barri','nom_barri','nom_districte','perfil_urbano','poblacion_total','total_incidencias_iris','incidencias_por_1000_hab','densidad_analitica_hab_ha','valor_catastral_unitario_eur_m2_mean']].rename(
    columns={'poblacion_total':'poblacion_2024','total_incidencias_iris':'incidencias_2024','incidencias_por_1000_hab':'incidencias_1000_2024','densidad_analitica_hab_ha':'densidad_2024'})

crecimiento=base15.merge(base24,on=['codi_barri','nom_barri','nom_districte','perfil_urbano'],how='inner')
crecimiento['crecimiento_poblacion_abs']=crecimiento['poblacion_2024']-crecimiento['poblacion_2015']; 
crecimiento['crecimiento_poblacion_pct']=(crecimiento['poblacion_2024']/crecimiento['poblacion_2015']-1)*100; 
crecimiento['crecimiento_incidencias_abs']=crecimiento['incidencias_2024']-crecimiento['incidencias_2015']; 
crecimiento['crecimiento_incidencias_pct']=(crecimiento['incidencias_2024']/crecimiento['incidencias_2015']-1)*100; 
crecimiento['crecimiento_tasa_incidencias_pct']=(crecimiento['incidencias_1000_2024']/crecimiento['incidencias_1000_2015']-1)*100; 
crecimiento['sube_mas_la_tasa_que_la_poblacion']=crecimiento['crecimiento_tasa_incidencias_pct']>crecimiento['crecimiento_poblacion_pct']
serie_ciudad=poblacion_ciudad.merge(df_analisis.groupby('any',as_index=False)['total_incidencias_iris'].sum().rename(columns={'total_incidencias_iris':'incidencias_barcelona'}),on='any',how='inner'); 
serie_ciudad['incidencias_por_1000_hab']=serie_ciudad['incidencias_barcelona']/serie_ciudad['poblacion_barcelona']*1000

## 9. Tipologías de incidencias

Resumo las áreas, elementos y detalles principales de IRIS para los gráficos del notebook 02.

In [10]:
# tipologias
inc=iris_limpio[(iris_limpio['tipus']=='INCIDENCIA')&iris_limpio['codi_barri'].notna()].copy(); inc['any']=pd.to_numeric(inc['any'],errors='coerce').astype('Int64')
for anio in [2024,2025]:
    ii=inc[inc['any']==anio].copy()
    ii.groupby(['codi_barri','barri','codi_districte','districte'],as_index=False).size().rename(columns={'size':f'total_incidencias_{anio}'}).to_csv(os.path.join(ANALISIS,f'incidencias_totales_barrio_{anio}.csv'),index=False)
    ra=ii['area'].value_counts(dropna=False).rename_axis('area').reset_index(name='casos'); ra['peso_pct']=(ra['casos']/ra['casos'].sum()*100).round(2); ra.to_csv(os.path.join(ANALISIS,f'resumen_areas_incidencias_{anio}.csv'),index=False)
    ii['element'].value_counts(dropna=False).head(15).rename_axis('element').reset_index(name='casos').to_csv(os.path.join(ANALISIS,f'resumen_elementos_incidencias_{anio}.csv'),index=False)
    det=ii['detall'].value_counts(dropna=False).head(10).rename_axis('detall').reset_index(name='casos')
    # area de cada detalle dominante
    det=det.merge(ii.groupby('detall')['area'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan).reset_index(),on='detall',how='left')
    det['porcentaje_total']=det['casos']/len(ii)*100; det.to_csv(os.path.join(ANALISIS,f'resumen_detalles_incidencias_{anio}.csv'),index=False)
inc2024=inc[inc['any']==2024].copy(); areas=pd.read_csv(os.path.join(ANALISIS,'resumen_areas_incidencias_2024.csv')).head(4)['area'].tolist()
iarea=inc2024[inc2024['area'].isin(areas)].groupby(['codi_barri','barri','districte','area'],as_index=False).size().rename(columns={'size':'casos'}); iarea.to_csv(os.path.join(ANALISIS,'incidencias_area_barrio_2024.csv'),index=False)
dom=iarea.loc[iarea.groupby('codi_barri')['casos'].idxmax()].copy(); dom.to_csv(os.path.join(ANALISIS,'area_dominante_barrio_2024.csv'),index=False)
if os.path.exists(os.path.join(INFORMES,'equidad_barrio_2022.csv')): pd.read_csv(os.path.join(INFORMES,'equidad_barrio_2022.csv')).to_csv(os.path.join(ANALISIS,'equidad_barrio_2022.csv'),index=False)

# 10. Proyección simple

Uso una regresión lineal sencilla para estimar la tendencia de incidencias. 
Primero valido el modelo contra 2024 y después incorporo 2024 al entrenamiento final para proyectar 2025 y 2026.
Excluyo 20, 21, 22 (interf COV)

In [11]:
# Serie anual de incidencias IRIS
serie_reg = (
    tabla2[tabla2['any'].between(2015, 2025)]
    .groupby('any', as_index=False)['total_incidencias_iris']
    .sum()
)

serie_reg['tipo_periodo'] = 'Observado'
serie_reg.loc[serie_reg['any'].isin([2015, 2016, 2017, 2018, 2019, 2023]), 'tipo_periodo'] = 'Usado para entrenar validación'
serie_reg.loc[serie_reg['any'].isin([2020, 2021, 2022]), 'tipo_periodo'] = 'Excluido entrenamiento'
serie_reg.loc[serie_reg['any'].eq(2024), 'tipo_periodo'] = 'Validación 2024'
serie_reg.loc[serie_reg['any'].eq(2025), 'tipo_periodo'] = 'Observado provisional'

serie_reg

,any,total_incidencias_iris,tipo_periodo
0,2015,54990.0,Usado para entrenar validación
1,2016,56723.0,Usado para entrenar validación
2,2017,66093.0,Usado para entrenar validación
3,2018,84811.0,Usado para entrenar validación
4,2019,92265.0,Usado para entrenar validación
5,2020,72934.0,Excluido entrenamiento
6,2021,87535.0,Excluido entrenamiento
7,2022,90802.0,Excluido entrenamiento
8,2023,119543.0,Usado para entrenar validación
9,2024,127122.0,Validación 2024


### Validación con 2024

Entreno una primera línea con 2015-2019 y 2023. Dejo 2024 fuera para comprobar cuánto se equivoca la tendencia respecto al dato observado.

In [12]:
# Entrenamiento inicial: excluyo 2020-2022 por ser años atípicos y dejo 2024 para validar.
train = serie_reg[serie_reg['any'].isin([2015, 2016, 2017, 2018, 2019, 2023])].copy()
coef, inter = np.polyfit(train['any'], train['total_incidencias_iris'], 1)
train['incidencias_estimadas_validacion'] = (coef * train['any'] + inter).round(0)

# R2 mide qué parte de la variación de los datos de entrenamiento explica la recta.
y_train = train['total_incidencias_iris']
y_pred_train = coef * train['any'] + inter
r2_validacion = 1 - ((y_train - y_pred_train) ** 2).sum() / ((y_train - y_train.mean()) ** 2).sum()

dato2024 = serie_reg.loc[serie_reg['any'].eq(2024), 'total_incidencias_iris'].iloc[0]
pred2024 = round(coef * 2024 + inter)
err = dato2024 - pred2024
errpct = err / dato2024 * 100

validacion_2024 = pd.DataFrame({
    'any': [2024],
    'dato_observado_2024': [int(dato2024)],
    'dato_estimado_2024': [int(pred2024)],
    'diferencia_observado_vs_estimado': [int(err)],
    'error_porcentual': [errpct],
    'r2_entrenamiento_validacion': [r2_validacion],
    'coeficiente_anual_validacion': [coef],
    'intercepto_validacion': [inter],
})

validacion_2024

,any,dato_observado_2024,dato_estimado_2024,diferencia_observado_vs_estimado,error_porcentual,r2_entrenamiento_validacion,coeficiente_anual_validacion,intercepto_validacion
0,2024,127122,130891,-3769,-2.964868,0.964752,8636.775,-1.734994e+07


### Entrenamiento final

Después de validar contra 2024, incorporo 2024 al entrenamiento final. Esta es la línea que uso para estimar 2025 y 2026.

In [13]:
# Entrenamiento final: incluye 2024 porque ya se usó para validar el comportamiento del modelo.
trainf = serie_reg[serie_reg['any'].isin([2015, 2016, 2017, 2018, 2019, 2023, 2024])].copy()
coeff, interf = np.polyfit(trainf['any'], trainf['total_incidencias_iris'], 1)

# R2 del modelo final usado para proyectar 2025 y 2026.
y_trainf = trainf['total_incidencias_iris']
y_pred_trainf = coeff * trainf['any'] + interf
r2_final = 1 - ((y_trainf - y_pred_trainf) ** 2).sum() / ((y_trainf - y_trainf.mean()) ** 2).sum()

serie_reg['incidencias_tendencia_final'] = (coeff * serie_reg['any'] + interf).round(0)
trainf = serie_reg[serie_reg['any'].isin([2015, 2016, 2017, 2018, 2019, 2023, 2024])].copy()
trainf['r2_modelo_final'] = r2_final

trainf

,any,total_incidencias_iris,tipo_periodo,incidencias_tendencia_final,r2_modelo_final
0,2015,54990.0,Usado para entrenar validación,53677.0,0.977151
1,2016,56723.0,Usado para entrenar validación,62040.0,0.977151
2,2017,66093.0,Usado para entrenar validación,70404.0,0.977151
3,2018,84811.0,Usado para entrenar validación,78767.0,0.977151
4,2019,92265.0,Usado para entrenar validación,87130.0,0.977151
8,2023,119543.0,Usado para entrenar validación,120583.0,0.977151
9,2024,127122.0,Validación 2024,128946.0,0.977151


### Proyección 2025-2026

Genero la tabla final de proyección. En 2025 sí existe un dato observado provisional, por eso se puede calcular la diferencia frente a la estimación; en 2026 todavía solo queda la estimación.

In [14]:
proy = pd.DataFrame({'any': [2025, 2026]})
proy['incidencias_estimadas'] = (coeff * proy['any'] + interf).round(0).astype(int)
proy['dato_observado_provisional'] = proy['any'].map(serie_reg.set_index('any')['total_incidencias_iris'])
proy['diferencia_observado_vs_estimado'] = proy['dato_observado_provisional'] - proy['incidencias_estimadas']
proy['coeficiente_anual'] = coeff
proy['intercepto'] = interf
proy['r2_modelo_final'] = r2_final
proy['error_validacion_2024'] = err
proy['error_porcentual_validacion_2024'] = errpct
proy['r2_entrenamiento_validacion'] = r2_validacion
proy['nota_2025'] = np.where(
    proy['any'].eq(2025),
    'Dato observado provisional: pendiente de confirmación por Open Data y posible efecto de límite de registros por usuario.',
    'Proyección sin dato observado disponible.'
)

proy

,any,incidencias_estimadas,dato_observado_provisional,diferencia_observado_vs_estimado,coeficiente_anual,intercepto,r2_modelo_final,error_validacion_2024,error_porcentual_validacion_2024,r2_entrenamiento_validacion,nota_2025
0,2025,137309,114281.0,-23028.0,8363.183468,-1.679814e+07,0.977151,-3769.0,-2.964868,0.964752,Dato observado provisional: pendiente de confi...
1,2026,145672,NaN,NaN,8363.183468,-1.679814e+07,0.977151,-3769.0,-2.964868,0.964752,Proyección sin dato observado disponible.


### Tendencia población-incidencias

Creo una línea auxiliar para representar la relación entre población e incidencias por barrio en 2024.

In [15]:
trend_data = df_2024[['poblacion_total', 'total_incidencias_iris']].dropna()
cp, ip = np.polyfit(trend_data['poblacion_total'], trend_data['total_incidencias_iris'], 1)

trend = pd.DataFrame({
    'poblacion_total': np.linspace(trend_data['poblacion_total'].min(), trend_data['poblacion_total'].max(), 200)
})
trend['incidencias_estimadas_linea'] = cp * trend['poblacion_total'] + ip

trend.head()

,poblacion_total,incidencias_estimadas_linea
0,841.000000,222.850502
1,1131.502513,242.469686
2,1422.005025,262.088870
3,1712.507538,281.708054
4,2003.010050,301.327237


## 12. Comprobación final

Al final reviso shapes y archivos principales. Si esta celda va bien, el notebook 02 debería poder ejecutarse.

In [16]:
# save common
poblacion_barrios.to_csv(os.path.join(ANALISIS,'poblacion_barrios_2015_2024.csv'),index=False); poblacion_ciudad.to_csv(os.path.join(ANALISIS,'poblacion_ciudad_2015_2024.csv'),index=False); df_analisis.to_csv(os.path.join(ANALISIS,'analisis_poblacion_densidad_2015_2024.csv'),index=False); df_2024.to_csv(os.path.join(ANALISIS,'analisis_poblacion_densidad_2024.csv'),index=False); crecimiento.to_csv(os.path.join(ANALISIS,'crecimiento_barrios_2015_2024.csv'),index=False); serie_ciudad.to_csv(os.path.join(ANALISIS,'serie_ciudad_poblacion_incidencias_2015_2024.csv'),index=False); serie_reg.to_csv(os.path.join(ANALISIS,'serie_regresion_incidencias_2015_2025.csv'),index=False); proy.to_csv(os.path.join(ANALISIS,'proyeccion_incidencias_2025_2026.csv'),index=False); trend.to_csv(os.path.join(ANALISIS,'tendencia_poblacion_incidencias_2024.csv'),index=False)
manifest=[]
for f in sorted(os.listdir(ANALISIS)):
    if f.endswith('.csv'):
        p=os.path.join(ANALISIS,f); manifest.append({'archivo':f,'filas':len(pd.read_csv(p)),'ruta':os.path.abspath(p)})
pd.DataFrame(manifest).to_csv(os.path.join(ANALISIS,'manifest_tablas_analisis.csv'),index=False)
print('FIN', tabla.shape, df_analisis.shape)

FIN (1022, 89) (730, 100)


In [17]:
archivos_necesarios = [
    'datos/processed/tabla_base_barrio_anio.csv',
    'datos/processed/analisis/analisis_poblacion_densidad_2024.csv',
    'datos/processed/analisis/serie_ciudad_poblacion_incidencias_2015_2024.csv',
    'datos/processed/analisis/proyeccion_incidencias_2025_2026.csv',
    'datos/processed/analisis/barrios_wgs84.geojson'
]

for archivo in archivos_necesarios:
    print(archivo, 'OK' if os.path.exists(archivo) else 'FALTA')

print('Tabla base:', pd.read_csv('datos/processed/tabla_base_barrio_anio.csv').shape)
print('Análisis 2024:', pd.read_csv('datos/processed/analisis/analisis_poblacion_densidad_2024.csv').shape)
print('Serie ciudad:', pd.read_csv('datos/processed/analisis/serie_ciudad_poblacion_incidencias_2015_2024.csv').shape)

datos/processed/tabla_base_barrio_anio.csv OK
datos/processed/analisis/analisis_poblacion_densidad_2024.csv OK
datos/processed/analisis/serie_ciudad_poblacion_incidencias_2015_2024.csv OK
datos/processed/analisis/proyeccion_incidencias_2025_2026.csv OK
datos/processed/analisis/barrios_wgs84.geojson OK
Tabla base: (1022, 89)
Análisis 2024: (73, 100)
Serie ciudad: (10, 4)
